# 🏗️ Building Age Classification from Satellite Imagery

# Notebook 1 — Introduction

---


## 1.1 Welcome

Welcome to the **Building Age Classification** hackathon challenge! 🎉

In this challenge, you will use Landsat satellite spectral features to classify the age of buildings in two European cities — Amsterdam and Madrid — into one of 4 age classes. The larger goal is to produce a classifier that can work on multiple cities with very little training data.

The hackathon dataset has been prepared from a larger pipeline that connects raw Landsat imagery to building construction records. All upstream processing (scene selection, downloading, spatial cropping, and pixel–building linking) has already been completed, so you can focus your time on the core machine learning problem.

This hackathon is structured into 4 notebooks:

| # | Notebook | Description |
|---|----------|-------------|
| **1** | **Introduction** *(you are here)* | Challenge overview, scientific context, and submission requirements |
| **2** | **Reading & Understanding the Data** | Load the dataset, explore columns, distributions, and data quality |
| **3** | **Preprocessing** | Feature engineering, class balancing, and data preparation |
| **4** | **Modelling & Evaluation** | Model training, evaluation, and prediction export |


---

## 1.2 The Challenge

**Task:** Given the spectral reflectance values of a 30 m × 30 m Landsat pixel that contains buildings, classify the majority age class (1–4) of the buildings in the pixel.

**Baseline input features:** 6 Landsat spectral bands — Blue, Green, Red, NIR, SWIR1, and SWIR2. These represent the minimum set of features available in the dataset. Teams are free (and encouraged) to engineer additional features, apply spatial context, or use any other strategy they see fit.

**Output:** A single integer per pixel — the predicted age class (1–4).


---

## 1.3 Background

Knowing when buildings were constructed is important for urban planning, infrastructure management, energy policy, and risk assessment. Many cities maintain construction year records, but these are often incomplete or inconsistent — especially across national borders.

The [Landsat programme](https://landsat.gsfc.nasa.gov/) (NASA / USGS) has been continuously imaging Earth's surface since 1972, making it one of the longest-running satellite observation programmes in history. Landsat satellites measure reflected sunlight in multiple wavelength bands (visible, near-infrared, short-wave infrared), producing a spectral fingerprint of whatever is on the ground. All Landsat data is freely and openly available.

The idea behind this challenge: building materials and construction practices have changed over time, and different materials reflect light differently across spectral bands. A model trained on pixels where the building age is known may be able to predict age where records are unavailable.

The key bottleneck here is obtaining the training data consisting of labeled pixels.  Extensive datasets exist for only a few cities (including Amsterdam and Madrid).  The main goal is to use the existing data to create a transferrable model--that is, one that can be retrained for a new city using only a small amount of data.  This is an example of *transfer learning*, in which a model trained on a large dataset is used as a starting point for a related task, in order to exploit learned representations without retraining from scratch.


---

## 1.4 The Age Classes

Buildings are grouped into **4 classes**. Two classes span the period covered by this dataset in equal 20-year intervals; two earlier classes are divided by a city-specific historical construction event whose exact year differs between Amsterdam and Madrid.

| Class | Period | Context |
|:-----:|:------:|:--------|
| **1** | Before [city event] | Pre-modern construction era; boundary is city-specific and will be provided by the organizers |
| **2** | [City event] to 1984 | Construction pre-dates the satellite record in this dataset; boundary is city-specific |
| **3** | 1984 – 2004 | First 20-year interval covered by this dataset |
| **4** | 2004 – 2024 | Second 20-year interval covered by this dataset |

Classes 3 and 4 use the same year boundaries for both cities. The exact years for Classes 1 and 2 will be provided by the organizers at the start of the hackathon.

> **Note on labels:** The provided data files include `weighted_mean_year` — the area-weighted mean construction year of the buildings overlapping each pixel. Age class labels are **not pre-assigned** in the files; you will derive them in Notebook 3 from `weighted_mean_year` using the class boundaries defined above.


---

## 1.5 Submission Requirements & Evaluation

### The evaluation procedure

You will receive the **Madrid training dataset** and the **Amsterdam transfer dataset**. Your task is to:

1. Use the Madrid data to train a **transferable model** for predicting age classes.
2. Use **cross-validation** on the Madrid data to obtain mean and standard deviation macro F1 score and confusion matrix proportions.
3. Develop a **transfer procedure** that takes the model from Step 1 and uses a small number of labeled Amsterdam pixels to produce a classification model for Amsterdam.
4. Use **cross-validation** on the Amsterdam transfer dataset to obtain mean and standard deviation **macro F1-scores** for Amsterdam models retrained with 5, 25, 50, 100, and 200 labeled Amsterdam pixels per class.

The organisers will evaluate your models on **held-out test sets** from both Amsterdam and Madrid that are not included in the provided data.

This design directly measures the core challenge: how well your approach adapts to a new city for different amounts of locally labeled data.

### What to submit

Four deliverables:

1. **Trained models** — your Stage 1 model (Madrid-trained)

2. **Training code** — a runnable notebook or script that adapts the Stage 1 model to Amsterdam using a small labeled sample, clearly parameterised so that the sample size per class (`n`) can be varied. Code must run end-to-end with a fixed random seed.

3. **PowerPoint presentation** — slides summarising your approach and results, including:
   - An abstract (max 150 words) on the first slide
   - Key design decisions and what worked / did not work
   - A table of the Madrid F1 score and four Amsterdam F1 scores (one per sample size) with error bars
   - A plot showing F1 (with error bars) vs. log2(sample size)

4. **Written justification** — max **300 words** covering your model design, transfer strategy, and interpretation of the four F1 scores

### Scoring

Four independent prize categories — a team can win more than one:

| Prize | Metric | What it rewards |
|-------|--------|----------------|
| **Best average F1** | Mean F1 across all five Amsterdam models | Consistent generalisation at every data volume |
| **Best peak F1** | Highest F1 across any single Amsterdam model | Best raw accuracy given enough target data |
| **Best low-data F1** | F1 of the 25-samples/class model only | Strongest generalisation with minimal labelling effort |
| **Best originality** | Judged on the abstract and presentation | Most novel or insightful approach to the cross-city problem |

The **low-data prize** is the hardest to win: it rewards models that have learned spectral representations that genuinely transfer across cities, rather than relying on large amounts of city-specific recalibration data.

The **originality prize** is judged qualitatively — it may go to a team with a lower F1 score whose approach is creative, well-reasoned, or opens an interesting direction.


---

## 1.6 The Dataset

The parquet files you will work with are the output of a multi-step pipeline. Here are the key facts about how the data was produced:

- **Satellites used:** Landsat 5 TM (1984–2013), Landsat 7 ETM+ (1999–present), Landsat 8 OLI (2013–present), Landsat 9 OLI-2 (2021–present).

- **Scene selection:** For each city and each year from 1984 to 2024, up to **3 candidate scenes** were selected from the summer months (June–August) and ranked by a combined score (`cloud_cover + missing_pixel_fraction`). Lower scores are better. All 3 candidates are retained in the dataset as separate observations, so each row may contain up to 3 spectral measurements for the same location in the same year.

- **Temporal sampling:** Scenes are retained for **every year** from 1984 onwards for which a usable scene could be found. There is no 5-year subsampling — the full annual time series is available.

- **Spatial cropping:** Raw Landsat scenes cover approximately 170 × 183 km. Each scene was clipped to the city bounding box before processing:
  - Amsterdam: roughly 35 × 35 km
  - Madrid: roughly 36 × 25 km

- **Pixel–building linking:** Building footprint data with construction year attributes was overlaid onto the 30 m Landsat grid. For each pixel, the pipeline computed:
  - `coverage`: fraction of pixel area overlapping building footprints
  - `weighted_mean_year`: area-weighted mean construction year of overlapping buildings

- **Wide-format layout:** Each row in the dataset represents one **geographic location** (a 30 m pixel) in one **year**. Up to 3 observations from that year are stored as column groups — `obs_1`, `obs_2`, `obs_3` — sorted by day-of-year. Each group contains the 6 spectral bands plus quality flags. A `qa_valid_N` boolean column marks whether each observation is a clean, cloud-free pixel.

- **Pre-filtering:** The dataset has already been cleaned. Only pixels with **≥ 15% building coverage** are included. Pixels with corrupt band values (zero or saturated) and construction years before 1850 have been removed.

- **Spectral bands:** Blue, Green, Red, NIR, SWIR1, SWIR2. Values are harmonised across sensor generations using the Roy et al. (2016) per-band linear correction (L5 TM → OLI equivalent), so all bands are on a consistent OLI reflectance scale.

- **No pre-assigned class labels:** The files include `weighted_mean_year` (continuous). You will assign age class labels in Notebook 3 by binning `weighted_mean_year` with `pd.cut()` using class boundaries provided by the organizers.


---

## 1.7 Cities: Amsterdam & Madrid

### 🇪🇸 Madrid

Madrid provides the **Stage 1 training data**. The city has comprehensive building age records and good Landsat coverage across all sensor eras. Your base model should be trained entirely on Madrid and must learn spectral features that are as city-independent as possible — that is the foundation of a generalisable system.

### 🇳🇱 Amsterdam

The **Amsterdam transfer dataset** is provided as your few-shot adaptation resource. You will use it to adapt your Madrid-trained model to Amsterdam by training five Stage 2 models, each using a different number of labeled samples per class (5,25,50,100,200).

### The broader goal: a city-portable model

The two-city setup is a proxy for a much larger ambition: **a building age classifier that can be deployed in any city in the world with minimal local labelling effort**.

Amsterdam and Madrid differ in climate (maritime vs. Mediterranean), building materials (brick and stone vs. concrete and stucco), urban density, and construction history. If your approach generalises well from Madrid to Amsterdam, it has a reasonable chance of working in other cities — Tokyo, São Paulo, Nairobi — without starting from scratch each time.

When designing your Stage 1 model, ask yourself: *does this feature or design choice help the model understand spectral signatures of age in general, or does it just memorise Madrid?* The answer to that question is the difference between a model that wins the low-data prize and one that does not.


---

## 1.8 What Comes Next

| Notebook | What you will do |
|----------|-----------------|
| **2 — Reading & Understanding the Data** | Load the parquet, explore features, labels, and data quality |
| **3 — Preprocessing** | Feature engineering, class balancing, and data preparation |
| **4 — Modelling & Evaluation** | Train your model, evaluate performance, and export predictions |

**Ready? Open Notebook 2 to start exploring the data.** 🚀
